# Notebook 01a — SimpleCNN Binary Baseline
**D7047E Advanced Deep Learning | Group 14**

Task: Binary — CLEAN (0) vs CROSSED-OUT (1)  
Model: SimpleCNN from scratch  
Run in **parallel** with `01_simplecnn_multiclass.ipynb`

WandB: `adl-crossouts-v3 / binary / SimpleCNN`

## 1. Configuration


In [ ]:
import sys
sys.path.insert(0, '..')
DATA_DIR       = '../dataset/iam_crossouts'
CHECKPOINT_DIR = '../checkpoints'
ZIP_PATH       = '../dataset/adl_dataset.zip'
FILE_ID        = '1dgIfz8aFwCuphLN9-L7gcU4QN0UQEKP3'
IMG_SIZE       = 224
BATCH_SIZE     = 64
NUM_WORKERS    = 16
LR             = 1e-3
EPOCHS         = 100
MIN_EPOCHS     = 20
PATIENCE       = 15
MODEL          = 'SimpleCNN'
print('Config loaded.')

## 2. Setup

In [ ]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb python-dotenv

In [ ]:
import os, zipfile, gc
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import torch, torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
import wandb
from dotenv import load_dotenv

from common import (get_transforms, WANDB_PROJECT, WANDB_GROUP_BINARY,
                    log_test_metrics, BinaryDataset, train_model, CATEGORIES)

WANDB_GROUP = WANDB_GROUP_BINARY

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 3. Dataset Download

In [ ]:
try:
    import gdown
except ImportError:
    import subprocess; subprocess.run(['pip','install','gdown','-q'],check=True); import gdown
os.makedirs(os.path.dirname(ZIP_PATH), exist_ok=True)
if not os.path.exists(ZIP_PATH):
    gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', ZIP_PATH, quiet=False)
if not os.path.exists(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH,'r') as zf: zf.extractall(DATA_DIR)
print('Dataset ready.')

## 4. Data Loaders

In [19]:
train_t, val_t = get_transforms(IMG_SIZE)

bin_train = BinaryDataset(os.path.join(DATA_DIR,'train','images'), train_t)
bin_val   = BinaryDataset(os.path.join(DATA_DIR,'val',  'images'), val_t)
bin_test  = BinaryDataset(os.path.join(DATA_DIR,'test', 'images'), val_t)

n_clean   = sum(1 for _,l in bin_train.samples if l==0)
n_crossed = sum(1 for _,l in bin_train.samples if l==1)
print(f'CLEAN: {n_clean:,}  CROSSED: {n_crossed:,}  Ratio 1:{n_crossed//n_clean}')
class_weights = torch.tensor([n_crossed/n_clean, 1.0]).to(device)

ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS>0 else None)
train_loader = DataLoader(bin_train, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(bin_val,   shuffle=False, **ldr_kw)
test_loader  = DataLoader(bin_test,  shuffle=False, **ldr_kw)
print(f'Train: {len(bin_train):,}  Val: {len(bin_val):,}  Test: {len(bin_test):,}')

CLEAN: 47,997  CROSSED: 383,976  Ratio 1:8
Train: 431,973  Val: 68,031  Test: 182,754


## 5. Model

In [20]:
from common import SimpleCNN

model = SimpleCNN(num_classes=2)
tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'SimpleCNN  trainable: {tr:,}')

SimpleCNN  trainable: 7,072,546


## 6. Train

In [ ]:
save_path = os.path.join(CHECKPOINT_DIR, f'best_binary_{MODEL}.pth')
resume_path = save_path if os.path.exists(save_path) else None
if resume_path: print(f'Resuming from {resume_path}')

model, history = train_model(
    MODEL, model, train_loader, val_loader,
    nn.CrossEntropyLoss(weight=class_weights),
    task='binary', save_path=save_path, device=device,
    lr=LR, epochs=EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
    wandb_project=WANDB_PROJECT, wandb_group=WANDB_GROUP, batch_size=BATCH_SIZE,
    resume_path=resume_path,
)

## 7. Test Evaluation

In [ ]:
model.eval()
preds, labels, probs = [], [], []
with torch.no_grad():
    for imgs, lbs in test_loader:
        out = model(imgs.to(device))
        probs.extend(torch.softmax(out,dim=1)[:,1].cpu().tolist())
        preds.extend(out.argmax(1).cpu().tolist())
        labels.extend(lbs.tolist())

print('Binary — SimpleCNN — Test Results')
print(f'  Accuracy:  {accuracy_score(labels, preds):.4f}')
print(f'  Precision: {precision_score(labels, preds, zero_division=0):.4f}')
print(f'  Recall:    {recall_score(labels, preds, zero_division=0):.4f}')
print(f'  F1:        {f1_score(labels, preds, zero_division=0):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(labels, probs):.4f}')

In [ ]:
log_test_metrics(
    preds=preds, labels=labels,
    probs=probs,
    model_name=MODEL,
    wandb_project=WANDB_PROJECT,
    wandb_group=WANDB_GROUP,
    task='binary',
)


## 8. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'],label='Train'); ax1.plot(history['val_loss'],label='Val')
ax1.set_title('SimpleCNN Binary — Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['train_acc'],label='Train'); ax2.plot(history['val_acc'],label='Val')
ax2.set_title('SimpleCNN Binary — Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout(); plt.savefig('binary_simplecnnv2_curves.png', dpi=150); plt.show()
print('Saved: binary_simplecnnv2_curves.png')
del model; torch.cuda.empty_cache(); gc.collect()

---
## Review
Before running `02_binary.ipynb`:
- [ ] Loss decreases smoothly, val tracks train
- [ ] OOM? → reduce `BATCH_SIZE` to 64
- [ ] Slow convergence? → lower `LR` to `5e-4`

Checkpoint: `checkpoints/best_binary_SimpleCNN.pth`